In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2

In [2]:
# Basic experiment settings
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)


cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Declare directories
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

# Experiment parameters
eff_img_size = 528
vit_img_size = 384
batch_size = 16
num_workers = 4
num_classes = 5
tta = True

# Currently testing vision transformers so check its architecture
vit_model = torch.load(
    "/kaggle/input/vit-v1-update/vit_v1_1.pt", map_location=device
).to(device)
eff_model = torch.load(
    "/kaggle/input/efficient-net/vit_cont_3.pt", map_location=device
).to(device)
linear_head = torch.load(
    "/kaggle/input/linear-cls/linear_cls.pt", map_location=device
).to(device)

cpu


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/vit-v1-update/vit_v1_1.pt'

In [3]:
# Test dataset without labels
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Args:
        data_dir: base directory to the images.
        img_label_df: dataframe containing image-label pairs.
        transforms: set of transforms to be used.
    """

    def __init__(
        self,
        data_dir,
        vit_size,
        efficient_size,
        transform=None,
        ttas=None,
        img_size=384,
    ):
        super().__init__(root=data_dir)

        self.transform = transform
        self.images = os.listdir(data_dir)
        self.ttas = ttas
        self.cc = v2.CenterCrop((600, 600))
        self.resize_vit = v2.Resize(
            (vit_size, vit_size), interpolation=InterpolationMode.BICUBIC
        )

        # Accidental error, change after linear cls updated
        self.resize_efficient = v2.Resize(
            (vit_size, vit_size), interpolation=InterpolationMode.BICUBIC
        )

    def __getitem__(self, idx):
        filename = self.images[idx]
        img = Image.open(os.path.join(self.root, filename))
        img = self.cc(img)
        vit_img = self.resize_vit(img)
        eff_img = self.resize_efficient(img)

        if self.ttas is not None and self.transform is not None:
            vit_img = [self.transform(t(vit_img)) for t in self.ttas]
            eff_img = [self.transform(t(eff_img)) for t in self.ttas]

        elif self.transform:
            vit_img = self.transform(vit_img)
            eff_img = self.transform(eff_img)

        return vit_img, eff_img, filename

    def __len__(self):
        return len(self.images)

In [4]:
# Create the set of transforms for test time
test_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

if tta:
    ttas = [
        v2.RandomRotation(180),
        v2.RandomVerticalFlip(1),
        v2.RandomPerspective(p=1),
    ]
else:
    ttas = None

# Dataset and dataloader
test_dataset = CassavaDataset(
    test_dir, vit_img_size, eff_img_size, transform=test_transforms, ttas=ttas
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

# Adjust unnormalized outputs
normalizer = torch.nn.Softmax(dim=1)

In [5]:
# Perform inference
all_names = []
all_preds = []

vit_model.eval()
eff_model.eval()
linear_head.eval()

with torch.no_grad():
    test_loss = 0

    for batch_idx, (vit_inputs, eff_inputs, filenames) in enumerate(test_loader):
        batch_size = len(filenames)

        if tta:
            vit_inputs, eff_inputs, filenames = (
                torch.cat(vit_inputs, dim=0).to(device),
                torch.cat(eff_inputs, dim=0).to(device),
                list(filenames),
            )

            vit_outputs = vit_model(vit_inputs)
            eff_outputs = eff_model(eff_inputs)

            # Find the average prediction logits by each model
            vit_batch_logits = torch.stack(torch.split(vit_outputs, batch_size), dim=0)
            vit_mean_logits = torch.mean(vit_batch_logits, dim=0)

            eff_batch_logits = torch.stack(torch.split(eff_outputs, batch_size), dim=0)
            eff_mean_logits = torch.mean(eff_batch_logits, dim=0)

            logit_inputs = torch.cat([vit_mean_logits, eff_mean_logits], dim=1)
            outputs = linear_head(logit_inputs)

            mean_preds = normalizer(outputs)
            pred_labels = torch.argmax(mean_preds, 1).tolist()

        else:
            # The same thing but without the troublesome stuff
            vit_inputs, eff_inputs, filenames = (
                vit_inputs.to(device),
                eff_inputs.to(device),
                list(filenames),
            )

            vit_outputs = vit_model(vit_inputs)
            eff_outputs = eff_model(eff_inputs)

            logit_inputs = torch.cat([vit_outputs, eff_outputs], dim=1)
            outputs = linear_head(logit_inputs)

            preds = normalizer(outputs)
            pred_labels = torch.argmax(preds, 1).tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)


NameError: name 'vit_model' is not defined

In [6]:
# Create submission
my_submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
my_submission.to_csv("submission.csv", index=False)
my_submission

,image_id,label
